# v2 Pipeline — Experiment Workbench

Self-contained (only `data/*.csv` needs to be on Drive). Trains the challenger
models and prints an **honest** scorecard.

**Every experiment is logged automatically** to `models_v2/experiments_log.csv`
on Drive (survives runtime restarts), plus a best-first `experiments_log.md`
leaderboard — so you can track progress and see exactly which config produced
the best model. View it any time with **Cell 6**.

## How to use
**First time / after restart:** run **Cell 1 → 2 → 3 → 4** in order.

| You changed... | Re-run |
|---|---|
| a **parameter** (model, coin, horizon, features, speed) in CONFIG | **Cell 3**, then **Cell 4** |
| the **engine code** (Cell 2) | **Cell 2**, then **Cell 4** |
| want the **leaderboard** of all runs so far | **Cell 6** |
| a quick **sweep** (horizons / coins / features) | **Cell 5** |
| ready to **deploy** (train the 6 final models) | **Cell 7** |

## Reading the result
`DIR` (50% = coin flip) · `p` < 0.05 = statistically real · `ERR vs persist`
must be **lower** to beat the naive baseline · `net` = bps **after** costs (must
be **positive** to be tradeable). `REAL EDGE: YES` needs all three.
**Only trust runs with `max_rows` blank (full history).** (Expect ~51-52% at 5-min.)

api.ipynb              → Run all          (get data)
v2_train_colab.ipynb   → Cells 1-4        (experiment), Cell 6 (leaderboard), Cell 7 (train live)
inference_orchestrator → Run all          (serve both model sets)
dashboard (Vercel)     → compare live

In [1]:
# ===== Cell 1: SETUP (run once per session) =====
%pip install -q ta scikit-learn torch joblib scipy pandas numpy

import os, sys, math, json, csv, argparse
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import ta
from datetime import datetime

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/CryptoProject'
except ImportError:
    BASE_DIR = os.path.abspath(os.getcwd())

DATA_DIR = os.path.join(BASE_DIR, 'data')
OUT_DIR  = os.path.join(BASE_DIR, 'models_v2')
os.makedirs(OUT_DIR, exist_ok=True)
assert os.path.isdir(DATA_DIR), f"data/ not found under {BASE_DIR} (put *_5m_data.csv there)."
print('BASE_DIR =', BASE_DIR, '| device =', 'cuda' if torch.cuda.is_available() else 'cpu')
print('CSV files:', sorted(f for f in os.listdir(DATA_DIR) if f.endswith('.csv')))
print('Experiment log ->', os.path.join(OUT_DIR, 'experiments_log.csv'))

Mounted at /content/drive
BASE_DIR = /content/drive/MyDrive/CryptoProject | device = cuda
CSV files: ['BTCUSDT_5m_data.csv', 'ETHUSDT_5m_data.csv', 'XRPUSDT_5m_data.csv']
Experiment log -> /content/drive/MyDrive/CryptoProject/models_v2/experiments_log.csv


In [2]:
# ===== Cell 2: ENGINE CODE (run once; re-run only if you edit it) =====

# ====== config constants (pipeline/config.py) ======
"""
Central configuration for the redesigned forecasting pipeline.

This pipeline is the staged successor to the live %B models. It is intentionally
kept SEPARATE from the production artifacts (best_lstm_model_*.pth, etc.) so the
currently-deployed engine is untouched until you deliberately promote new models.

Key design changes vs. the legacy pipeline (see audit_report.md):
  * Target is the H-step log-return, not next-step Bollinger %B.
  * A classification head (UP / FLAT / DOWN with a volatility dead-band) is
    trained jointly with the return regressor, so the model optimizes the thing
    we actually deploy (price direction).
  * Order-flow features (taker buy/sell imbalance, trade intensity/size) that
    were already collected but unused are added.
  * Volume / order-flow normalization is CAUSAL (rolling), removing the global
    look-ahead leakage in the legacy preprocessing.
  * Walk-forward (rolling-origin) validation replaces the single chronological
    split, giving an honest out-of-sample estimate.
"""


# Staged artifacts go here, NOT in models/, so production is never overwritten.

SYMBOLS = ["BTCUSDT", "ETHUSDT", "XRPUSDT"]

CONFIG = {
    # --- target / horizon ---
    "horizon": 1,          # candles ahead (1=5m, 3=15m, 6=30m, 12=60m). Sweep this.
    "seq_length": 60,      # input window length (candles)
    "deadband_k": 0.33,    # FLAT if |return| < deadband_k * rolling return-vol
    "vol_window": 288,     # ~1 day of 5m candles, for causal vol / normalization
    # --- training ---
    "batch_size": 128,
    "epochs": 50,
    "patience": 6,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "cls_loss_weight": 1.0,   # weight of classification loss vs. regression (Huber)
    "dropout": 0.2,
    # --- model sizes ---
    "lstm_hidden": 64,
    "tft_d_model": 32,
    "tft_heads": 4,
    "tft_layers": 2,
    # --- walk-forward ---
    "wf_folds": 5,         # number of rolling test folds
    "wf_test_frac": 0.10,  # each test fold = this fraction of the series
    "wf_val_frac": 0.10,   # validation block right before each test fold
    # --- fast-experiment knobs (cut run time when you're only checking direction) ---
    "max_rows": None,      # cap to most recent N candles (None = full history). e.g. 150000 ~ last ~1.4yr
    "train_step": 1,       # stride for TRAINING windows (1=all; 4 => 4x fewer train samples, val/test stay 1)
    "pred_batch": 4096,    # batch size for inference (fixes the seq_length=120 OOM)
    # --- feature set ---
    "use_extra_features": False,  # True => add the FEATURES_EXTRA block (volatility / order-flow / MTF)
    # --- smoke test (tiny run on local cached CSV to prove the code executes) ---
    "smoke": False,
}

# Base + order-flow feature set. Everything is causal (no future leakage).
FEATURES_BASE = [
    "log_ret", "rsi", "rsi_change", "rsi_accel",
    "macd", "macd_slope", "bb_pband_change", "ma_dist",
    "volume_z", "vol_spike", "adx",
    "hour_sin", "hour_cos", "mom_3", "mom_5",
    # --- order flow ---
    "taker_buy_imb",   # 2*taker_buy_frac - 1  in [-1,1]; >0 = aggressive buying
    "trade_intensity", # causal z-score of log(number_of_trades)
    "trade_size",      # causal z-score of log(quote_volume / number_of_trades)
]

# Optional extra block (enabled via cfg['use_extra_features']) — volatility,
# sustained order-flow, multi-timeframe trend, candle geometry. The lever to
# test whether richer features help (audit improvement #6). All causal.
FEATURES_EXTRA = [
    "rv",          # realized vol (1h rolling std of log_ret), causal z-scored
    "atr_pct",     # ATR(14) / close * 100
    "buy_imb_ma",  # sustained taker-buy imbalance (rolling mean)
    "cvd_z",       # cumulative volume delta over a window, causal z-scored
    "trend_mtf",   # multi-timeframe trend: (EMA12 - EMA48)/close
    "range_pos",   # where close sits in the candle range [0,1]
    "dist_hi20",   # distance to 20-bar high (breakout proximity)
    "dist_lo20",   # distance to 20-bar low
]

# Backward-compatible default (the base set).
FEATURES = FEATURES_BASE

def feature_list(cfg):
    """The feature columns a run uses, per cfg['use_extra_features']."""
    return FEATURES_BASE + FEATURES_EXTRA if cfg.get("use_extra_features") else FEATURES_BASE

CLASS_NAMES = ["DOWN", "FLAT", "UP"]  # indices 0,1,2


# ====== features (pipeline/features.py) ======
"""
Feature engineering — single source of truth for BOTH training and inference.

Every transform here is CAUSAL: a feature at time t uses only data up to and
including t. The legacy pipeline normalized volume with a global mean/std over
the whole series (including the test period) — that look-ahead leakage is fixed
here by using rolling statistics.
"""

import numpy as np
import pandas as pd
import ta


EPS = 1e-9


def _causal_z(series: pd.Series, window: int) -> pd.Series:
    """Rolling z-score using only past+current values (no future leakage)."""
    mean = series.rolling(window, min_periods=window // 4).mean()
    std = series.rolling(window, min_periods=window // 4).std()
    return (series - mean) / (std + EPS)


def compute_features(df: pd.DataFrame, vol_window: int = None) -> pd.DataFrame:
    """Add all model features + Bollinger bands (kept for reference/plots).
    Expects raw OHLCV + order-flow columns from the Binance kline CSV."""
    vol_window = vol_window or CONFIG["vol_window"]
    df = df.copy()
    df["open_time"] = pd.to_datetime(df["open_time"])
    df = df.sort_values("open_time").drop_duplicates("open_time").reset_index(drop=True)

    c = df["close"]
    df["log_ret"] = np.log(c / c.shift(1))

    df["rsi"] = ta.momentum.rsi(c, window=14) / 100.0
    df["rsi_change"] = df["rsi"].diff(3)
    df["rsi_accel"] = df["rsi_change"].diff(2)

    macd = ta.trend.MACD(c)
    macd_raw = macd.macd_diff()
    df["macd"] = (macd_raw - macd_raw.rolling(100).mean()) / (macd_raw.rolling(100).std() + EPS)
    df["macd_slope"] = macd.macd_diff().diff(2)

    bb = ta.volatility.BollingerBands(c, window=20, window_dev=2)
    df["bb_pband"] = bb.bollinger_pband()
    df["bb_pband_change"] = df["bb_pband"].diff(1)
    df["bb_hband"] = bb.bollinger_hband()
    df["bb_lband"] = bb.bollinger_lband()

    df["ma_20"] = c.rolling(20).mean()
    df["ma_dist"] = (c - df["ma_20"]) / (df["ma_20"] + EPS) * 10.0

    # --- Volume (CAUSAL z-score, was global-leaky before) ---
    log_vol = np.log(df["volume"] + 1)
    df["volume_z"] = _causal_z(log_vol, vol_window)
    vol_ma = log_vol.rolling(20).mean()
    df["vol_spike"] = (log_vol > (vol_ma * 2)).astype(float)

    adx = ta.trend.ADXIndicator(df["high"], df["low"], c, window=14)
    df["adx"] = adx.adx()

    hour = df["open_time"].dt.hour
    df["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    df["hour_cos"] = np.cos(2 * np.pi * hour / 24)

    df["mom_3"] = c / c.shift(3) - 1
    df["mom_5"] = c / c.shift(5) - 1

    # --- Order-flow features (NEW: previously collected but unused) ---
    taker_buy = df.get("taker_buy_base_asset_volume")
    if taker_buy is not None:
        taker_buy = pd.to_numeric(taker_buy, errors="coerce")
        buy_frac = (taker_buy / (df["volume"] + EPS)).clip(0, 1)
        df["taker_buy_imb"] = 2 * buy_frac - 1.0          # [-1,1]
    else:
        df["taker_buy_imb"] = 0.0

    n_trades = pd.to_numeric(df.get("number_of_trades", np.nan), errors="coerce")
    df["trade_intensity"] = _causal_z(np.log(n_trades + 1), vol_window)

    quote_vol = pd.to_numeric(df.get("quote_asset_volume", np.nan), errors="coerce")
    avg_trade = np.log(quote_vol / (n_trades + EPS) + 1)
    df["trade_size"] = _causal_z(avg_trade, vol_window)

    # --- Extra features (used only when cfg['use_extra_features']; always computed
    #     here so inference can select whatever a model's meta lists). All causal. ---
    df["rv"] = _causal_z(df["log_ret"].rolling(12).std(), vol_window)
    tr = pd.concat([(df["high"] - df["low"]),
                    (df["high"] - c.shift()).abs(),
                    (df["low"] - c.shift()).abs()], axis=1).max(axis=1)
    df["atr_pct"] = (tr.rolling(14).mean() / (c + EPS)) * 100
    df["buy_imb_ma"] = df["taker_buy_imb"].rolling(12).mean()
    signed_vol = df["taker_buy_imb"] * np.log(df["volume"] + 1)
    df["cvd_z"] = _causal_z(signed_vol.rolling(48).sum(), vol_window)
    ema_f = c.ewm(span=12, adjust=False).mean()
    ema_s = c.ewm(span=48, adjust=False).mean()
    df["trend_mtf"] = (ema_f - ema_s) / (c + EPS) * 100
    rng = (df["high"] - df["low"])
    df["range_pos"] = ((c - df["low"]) / (rng + EPS)).clip(0, 1)
    df["dist_hi20"] = (c - df["high"].rolling(20).max()) / (c + EPS) * 100
    df["dist_lo20"] = (c - df["low"].rolling(20).min()) / (c + EPS) * 100

    return df


def make_targets(df: pd.DataFrame, horizon: int, deadband_k: float,
                 vol_window: int) -> pd.DataFrame:
    """Add the regression target (H-step log-return) and the 3-class label.

    target_ret  = log(close[t+H] / close[t])
    label       = UP   if ret >  +k*sigma_t
                  DOWN if ret <  -k*sigma_t
                  FLAT otherwise
    where sigma_t is a CAUSAL rolling std of 1-step log-returns scaled to the
    horizon (sqrt-time). The dead-band prevents the model from being graded on
    microscopic, untradeable moves.
    """
    df = df.copy()
    fwd_ret = np.log(df["close"].shift(-horizon) / df["close"])
    df["target_ret"] = fwd_ret

    sigma1 = df["log_ret"].rolling(vol_window, min_periods=vol_window // 4).std()
    sigma_h = sigma1 * np.sqrt(horizon)
    thr = deadband_k * sigma_h

    label = np.full(len(df), 1, dtype=float)  # FLAT
    label[fwd_ret > thr] = 2                   # UP
    label[fwd_ret < -thr] = 0                  # DOWN
    label[fwd_ret.isna() | thr.isna()] = np.nan
    df["target_cls"] = label
    return df


# ====== dataset + walk-forward (pipeline/dataset.py) ======
"""
Sequence building, causal scaling, and walk-forward (rolling-origin) splits.

Walk-forward replaces the legacy single 80/10/10 chronological split: we slide a
train/val/test window across time and aggregate out-of-sample test predictions,
which is the honest way to estimate live performance and to detect drift.
"""

import os
import numpy as np
import pandas as pd



def load_frame(symbol: str, cfg: dict) -> pd.DataFrame:
    """Load raw CSV -> features -> targets -> drop warm-up/no-target rows.
    cfg['max_rows'] caps to the most recent N candles (fast experiments)."""
    feats = feature_list(cfg)
    path = os.path.join(DATA_DIR, f"{symbol}_5m_data.csv")
    df = pd.read_csv(path)
    df = compute_features(df, vol_window=cfg["vol_window"])
    df = make_targets(df, cfg["horizon"], cfg["deadband_k"], cfg["vol_window"])
    keep = feats + ["target_ret", "target_cls", "close", "open_time",
                    "bb_lband", "bb_hband"]
    df = df[keep].dropna().reset_index(drop=True)
    if cfg.get("max_rows"):
        df = df.tail(int(cfg["max_rows"])).reset_index(drop=True)
    return df


def _sequences(feat_scaled, ret, cls, base_close, seq_len, step=1):
    """Window the rows. y is aligned to the LAST candle of each window, whose
    forward return / class is the supervised target. base_close is that candle's
    close (the price the prediction is made from)."""
    X, yr, yc, bc = [], [], [], []
    for i in range(0, len(feat_scaled) - seq_len + 1, step):
        j = i + seq_len - 1
        X.append(feat_scaled[i:i + seq_len])
        yr.append(ret[j]); yc.append(cls[j]); bc.append(base_close[j])
    return (np.asarray(X, dtype=np.float32), np.asarray(yr, dtype=np.float32),
            np.asarray(yc, dtype=np.int64), np.asarray(bc, dtype=np.float64))


def walk_forward_folds(df: pd.DataFrame, cfg: dict):
    """Yield dicts with scaled train/val/test sequences for each rolling fold.
    StandardScaler is fit on each fold's TRAIN slice only (no leakage)."""
    from sklearn.preprocessing import StandardScaler

    feats = feature_list(cfg)
    n = len(df)
    seq = cfg["seq_length"]
    train_step = max(1, int(cfg.get("train_step", 1)))
    test_sz = int(n * cfg["wf_test_frac"])
    val_sz = int(n * cfg["wf_val_frac"])
    folds = cfg["wf_folds"]
    if test_sz < seq + 5 or val_sz < seq + 5:
        folds = 1  # tiny data (e.g. smoke test): one fold

    feat = df[feats].values
    ret = df["target_ret"].values
    cls = df["target_cls"].values.astype(np.int64)
    base = df["close"].values
    times = df["open_time"].values

    # Place test folds at the end of the series, sliding backwards.
    for k in range(folds):
        test_end = n - k * test_sz
        test_start = test_end - test_sz
        val_start = test_start - val_sz
        train_end = val_start
        if train_end < seq + 10:
            break
        sc = StandardScaler().fit(feat[:train_end])
        sl = lambda a, b: slice(max(0, a), b)

        def build(a, b, step=1):
            return _sequences(sc.transform(feat[sl(a, b)]), ret[sl(a, b)],
                              cls[sl(a, b)], base[sl(a, b)], seq, step=step)

        def daterange(a, b):
            s = times[sl(a, b)]
            return (str(s[0])[:10], str(s[-1])[:10]) if len(s) else ("-", "-")

        yield {
            "fold": k,
            "scaler": sc,
            "train": build(0, train_end, train_step),  # subsample TRAIN only
            "val": build(val_start, test_start),       # val/test stay step=1 (honest)
            "test": build(test_start, test_end),
            "test_time": times[sl(test_start, test_end)][seq - 1:],
            "dates": {"train": daterange(0, train_end),
                      "val": daterange(val_start, test_start),
                      "test": daterange(test_start, test_end)},
        }


# ====== models (pipeline/models.py) ======
"""
Dual-head models: each outputs (regression of H-step log-return, 3-class logits).
Architectures mirror the production LSTM (attention+MLP) and TFT (VSN+encoder)
so lessons transfer, but with a classification head added.
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class _Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, lstm_out):
        w = F.softmax(self.attention(lstm_out), dim=1)
        return torch.sum(w * lstm_out, dim=1)


class LSTMDual(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2, n_classes=3):
        super().__init__()
        self.hidden_dim = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.attn = _Attention(hidden_size)
        self.shared = nn.Sequential(nn.Linear(hidden_size, hidden_size // 2), nn.ReLU(), nn.Dropout(dropout))
        self.reg_head = nn.Linear(hidden_size // 2, 1)
        self.cls_head = nn.Linear(hidden_size // 2, n_classes)

    def forward(self, x):
        h0 = x.new_zeros(self.num_layers, x.size(0), self.hidden_dim)
        c0 = x.new_zeros(self.num_layers, x.size(0), self.hidden_dim)
        out, _ = self.lstm(x, (h0, c0))
        z = self.shared(self.attn(out))
        return self.reg_head(z).squeeze(-1), self.cls_head(z)


# ---- TFT components (same as production) ----
class _GLU(nn.Module):
    def __init__(self, size):
        super().__init__()
        self.fc = nn.Linear(size, size * 2)

    def forward(self, x):
        a, b = torch.chunk(self.fc(x), 2, dim=-1)
        return a * torch.sigmoid(b)


class _GRN(nn.Module):
    def __init__(self, in_size, hidden, out_size=None, dropout=0.1):
        super().__init__()
        out_size = out_size or in_size
        self.fc1 = nn.Linear(in_size, hidden)
        self.fc2 = nn.Linear(hidden, out_size)
        self.glu = _GLU(out_size)
        self.ln = nn.LayerNorm(out_size)
        self.drop = nn.Dropout(dropout)
        self.skip = nn.Linear(in_size, out_size) if in_size != out_size else nn.Identity()

    def forward(self, x):
        res = self.skip(x)
        x = self.fc2(torch.relu(self.fc1(x)))
        return self.ln(res + self.glu(self.drop(x)))


class _VSN(nn.Module):
    def __init__(self, input_dim, num_vars, d_model, dropout=0.1):
        super().__init__()
        self.num_vars = num_vars
        self.grns = nn.ModuleList([_GRN(input_dim // num_vars, d_model, d_model, dropout)
                                   for _ in range(num_vars)])
        self.selector = _GRN(input_dim, d_model, num_vars, dropout)

    def forward(self, x):
        w = torch.softmax(self.selector(x), dim=-1)
        chunk = x.shape[-1] // self.num_vars
        outs = [self.grns[i](x[..., i * chunk:(i + 1) * chunk]) for i in range(self.num_vars)]
        outs = torch.stack(outs, dim=-1)
        return torch.sum(outs * w.unsqueeze(-2), dim=-1)


class _PosEnc(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class TFTDual(nn.Module):
    def __init__(self, num_vars, d_model=32, nhead=4, num_layers=2, dropout=0.2, n_classes=3):
        super().__init__()
        self.vsn = _VSN(num_vars, num_vars, d_model, dropout)
        self.pos = _PosEnc(d_model)
        layer = nn.TransformerEncoderLayer(d_model, nhead, d_model * 4, dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers)
        self.reg_head = nn.Linear(d_model, 1)
        self.cls_head = nn.Linear(d_model, n_classes)

    def forward(self, x):
        z = self.encoder(self.pos(self.vsn(x)))[:, -1, :]
        return self.reg_head(z).squeeze(-1), self.cls_head(z)


# ====== trainer + auto-logging + honest eval (pipeline/train.py) ======
"""
Walk-forward trainer for the dual-head (return + direction) models.

For each coin x model:
  * train with joint loss = Huber(return) + w * CrossEntropy(class) over every
    walk-forward fold;
  * collect OUT-OF-SAMPLE test predictions across all folds;
  * score them honestly (price direction + significance + cost-aware edge,
    reusing eval_harness), comparing to a persistence baseline;
  * save the model trained on the most-recent fold + its scaler + metadata to
    models_v2/ (production models in models/ are never touched).

Run:  python -m pipeline.train --model lstm --symbol BTCUSDT
      python -m pipeline.train --all
      python -m pipeline.train --smoke           # tiny end-to-end run on local CSV
"""

import os
import sys
import csv
import json
import argparse
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datetime import datetime



try:
    from scipy.stats import binomtest
    def _binom_p(k, n):
        return binomtest(k, n, 0.5, alternative="two-sided").pvalue if n else float("nan")
except Exception:
    def _binom_p(k, n):
        return float("nan")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def _make_model(model_type, n_features, cfg):
    if model_type == "lstm":
        return LSTMDual(n_features, cfg["lstm_hidden"], 2, cfg["dropout"])
    return TFTDual(n_features, cfg["tft_d_model"], cfg["tft_heads"],
                   cfg["tft_layers"], cfg["dropout"])


def _loader(arrs, bs, shuffle):
    X, yr, yc, bc = arrs
    ds = torch.utils.data.TensorDataset(
        torch.tensor(X), torch.tensor(yr), torch.tensor(yc))
    return torch.utils.data.DataLoader(ds, batch_size=bs, shuffle=shuffle)


def _train_one_fold(model, fold, cfg):
    huber = nn.HuberLoss(delta=1.0)
    ce = nn.CrossEntropyLoss()
    w = cfg["cls_loss_weight"]
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=3)

    tr = _loader(fold["train"], cfg["batch_size"], True)
    va = _loader(fold["val"], cfg["batch_size"], False)
    best, best_state, bad = np.inf, None, 0

    for epoch in range(cfg["epochs"]):
        model.train()
        for X, yr, yc in tr:
            X, yr, yc = X.to(DEVICE), yr.to(DEVICE), yc.to(DEVICE)
            opt.zero_grad()
            pr, pc = model(X)
            loss = huber(pr, yr) + w * ce(pc, yc)
            loss.backward(); opt.step()
        # validation
        model.eval(); vloss = 0.0; nb = 0
        with torch.no_grad():
            for X, yr, yc in va:
                X, yr, yc = X.to(DEVICE), yr.to(DEVICE), yc.to(DEVICE)
                pr, pc = model(X)
                vloss += (huber(pr, yr) + w * ce(pc, yc)).item(); nb += 1
        vloss = vloss / max(nb, 1)
        sched.step(vloss)
        if vloss < best - 1e-7:
            best, best_state, bad = vloss, {k: v.cpu().clone() for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= cfg["patience"]:
                break
    if best_state:
        model.load_state_dict(best_state)
    return model, best


def _predict(model, arrs, batch_size=4096):
    """Batched inference. The old version pushed the whole test set to the GPU in
    one forward pass, which OOMs on large test folds / seq_length=120."""
    X = arrs[0]
    model.eval()
    prs, pcs = [], []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            xb = torch.tensor(X[i:i + batch_size]).to(DEVICE)
            pr, pc = model(xb)
            prs.append(pr.cpu().numpy())
            pcs.append(pc.softmax(-1).cpu().numpy())
    return np.concatenate(prs), np.concatenate(pcs)


def _evaluate(pred_ret, prob_cls, fold_arrs, cost_bps=2.0):
    """Honest OOS scoring on a fold's test set.
    Direction from the classifier (argmax over DOWN/UP, FLAT = no trade).
    Reconstructed price = base * exp(pred_ret); ERR vs persistence (=base)."""
    _, yr, yc, base = fold_arrs
    actual_ret = yr
    actual_dir = np.sign(actual_ret)
    cls = prob_cls.argmax(1)             # 0 DOWN,1 FLAT,2 UP
    pred_dir = np.where(cls == 2, 1, np.where(cls == 0, -1, 0))

    trade = pred_dir != 0
    m = trade & (actual_dir != 0)
    hits = int(((pred_dir[m] > 0) == (actual_dir[m] > 0)).sum())
    n_dir = int(m.sum())
    dir_acc = hits / n_dir * 100 if n_dir else float("nan")

    recon = base * np.exp(pred_ret)
    actual_price = base * np.exp(actual_ret)
    err = np.mean(np.abs(recon - actual_price) / actual_price) * 100
    persist_err = np.mean(np.abs(base - actual_price) / actual_price) * 100

    # cost-aware edge: signed realized return on traded bars, minus round-trip cost
    signed = pred_dir[m] * actual_ret[m]
    gross_bps = float(np.mean(signed) * 1e4) if n_dir else float("nan")
    return {
        "n_test": len(yr), "n_trades": n_dir,
        "dir_acc": dir_acc, "dir_hits": hits, "binom_p": _binom_p(hits, n_dir),
        "err": float(err), "persist_err": float(persist_err),
        "gross_bps": gross_bps, "net_bps": gross_bps - cost_bps,
        "frac_flat": float(np.mean(cls == 1)),
    }


LOG_COLUMNS = [
    "timestamp", "model", "symbol", "horizon_min", "n_features", "use_extra",
    "seq_length", "max_rows", "wf_folds", "epochs", "train_step",
    "deadband_k", "cls_w", "lr", "dropout",
    "DIR", "p", "trades", "ERR", "persist_ERR", "net_bps", "real_edge",
]


def _log_experiment(cfg, model_type, symbol, feats, s):
    """Append one row to models_v2/experiments_log.csv (on Drive, so it survives
    runtime restarts) and regenerate a best-first leaderboard experiments_log.md.
    Runs automatically from run(), so every experiment is captured with its full
    config -> you can track progress and see exactly what produced the best model."""
    os.makedirs(OUT_DIR, exist_ok=True)
    log_csv = os.path.join(OUT_DIR, "experiments_log.csv")
    row = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "model": model_type, "symbol": symbol, "horizon_min": cfg["horizon"] * 5,
        "n_features": len(feats), "use_extra": int(bool(cfg.get("use_extra_features"))),
        "seq_length": cfg["seq_length"], "max_rows": cfg.get("max_rows"),
        "wf_folds": cfg["wf_folds"], "epochs": cfg["epochs"],
        "train_step": cfg.get("train_step", 1), "deadband_k": cfg["deadband_k"],
        "cls_w": cfg["cls_loss_weight"], "lr": cfg["lr"], "dropout": cfg["dropout"],
        "DIR": round(s["dir"], 2), "p": round(s["p"], 4), "trades": s["trades"],
        "ERR": round(s["err"], 4), "persist_ERR": round(s["persist"], 4),
        "net_bps": (round(s["net"], 2) if s["net"] == s["net"] else ""),  # nan -> blank
        "real_edge": s["edge"],
    }
    exists = os.path.exists(log_csv)
    with open(log_csv, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=LOG_COLUMNS)
        if not exists:
            w.writeheader()
        w.writerow(row)
    # rebuild a sorted, human-readable leaderboard (no external deps)
    try:
        df = pd.read_csv(log_csv)
        df["_edge"] = (df["real_edge"] == "YES").astype(int)
        df["_net"] = pd.to_numeric(df["net_bps"], errors="coerce").fillna(-9999)
        df = df.sort_values(["_edge", "_net", "DIR"], ascending=False).drop(columns=["_edge", "_net"])
        full = df[df["max_rows"].isna()]
        hdr = list(df.columns)
        def table(d):
            out = ["| " + " | ".join(hdr) + " |", "|" + "|".join(["---"] * len(hdr)) + "|"]
            for _, r in d.iterrows():
                out.append("| " + " | ".join("" if pd.isna(r[h]) else str(r[h]) for h in hdr) + " |")
            return "\n".join(out)
        md = [f"# Experiments leaderboard (auto-generated)\n",
              f"_{len(df)} runs, updated {row['timestamp']}. Sorted best-first: "
              f"REAL EDGE, then net bps, then DIR._\n",
              "## Full-history runs (max_rows blank = decisive)\n",
              (table(full) if len(full) else "_none yet — set MAX_ROWS=None for a decisive run._"),
              "\n## All runs (incl. fast/partial-window probes)\n", table(df)]
        with open(os.path.join(OUT_DIR, "experiments_log.md"), "w") as f:
            f.write("\n".join(md))
    except Exception as e:
        print(f"   (leaderboard rebuild skipped: {e})")
    print(f"   logged -> {log_csv}  ({'appended' if exists else 'created'}; run #{_count(log_csv)})")


def _count(path):
    try:
        with open(path) as f:
            return sum(1 for _ in f) - 1
    except Exception:
        return "?"


def run(model_type, symbol, cfg):
    feats = feature_list(cfg)
    print(f"\n{'='*70}\n{symbol}  {model_type.upper()}  "
          f"(horizon={cfg['horizon']*5}m, seq={cfg['seq_length']}, "
          f"features={len(feats)}{'+extra' if cfg.get('use_extra_features') else ''}, "
          f"train_step={cfg.get('train_step',1)}, folds={cfg['wf_folds']}, "
          f"max_rows={cfg.get('max_rows')})\n{'='*70}")
    df = load_frame(symbol, cfg)
    print(f"  data: {len(df)} rows  {str(df['open_time'].iloc[0])[:10]} -> {str(df['open_time'].iloc[-1])[:10]}")
    cls_counts = df["target_cls"].value_counts().to_dict()
    print(f"  class balance: " + ", ".join(f"{CLASS_NAMES[int(k)]}={int(v)}" for k, v in sorted(cls_counts.items())))

    pred_batch = int(cfg.get("pred_batch", 4096))
    agg, last_fold = [], None
    for fold in walk_forward_folds(df, cfg):
        if len(fold["train"][0]) < 20 or len(fold["test"][0]) < 5:
            continue
        model = _make_model(model_type, len(feats), cfg).to(DEVICE)
        model, vloss = _train_one_fold(model, fold, cfg)
        pr, pc = _predict(model, fold["test"], batch_size=pred_batch)
        res = _evaluate(pr, pc, fold["test"])
        res["fold"] = fold["fold"]; res["val_loss"] = vloss
        agg.append(res); last_fold = (model, fold)
        d = fold["dates"]
        print(f"  fold {fold['fold']}: train {d['train'][0]}..{d['train'][1]} "
              f"test {d['test'][0]}..{d['test'][1]} | "
              f"n={res['n_test']} trades={res['n_trades']} "
              f"DIR={res['dir_acc']:5.1f}% (p={res['binom_p']:.3f}) "
              f"ERR={res['err']:.4f}% vs persist {res['persist_err']:.4f}% "
              f"net={res['net_bps']:+.1f}bps")

    if not agg:
        print("  !! not enough data for any fold")
        return None

    # pooled OOS summary
    tot_hits = sum(r["dir_hits"] for r in agg)
    tot_trades = sum(r["n_trades"] for r in agg)
    pooled_dir = tot_hits / tot_trades * 100 if tot_trades else float("nan")
    pooled_p = _binom_p(tot_hits, tot_trades)
    mean_err = np.mean([r["err"] for r in agg])
    mean_persist = np.mean([r["persist_err"] for r in agg])
    mean_net = np.nanmean([r["net_bps"] for r in agg])  # folds with 0 trades are nan
    beats = "YES" if (pooled_p < 0.05 and pooled_dir > 50 and mean_err < mean_persist) else "no"
    print(f"  --> POOLED OOS: DIR={pooled_dir:.1f}% (p={pooled_p:.3f}, {tot_hits}/{tot_trades}) "
          f"ERR={mean_err:.4f}% vs persist {mean_persist:.4f}%  net={mean_net:+.1f}bps  "
          f"REAL EDGE: {beats}")

    # automatic experiment log (config + result) -> Drive
    _log_experiment(cfg, model_type, symbol, feats,
                    {"dir": pooled_dir, "p": pooled_p, "trades": tot_trades,
                     "err": mean_err, "persist": mean_persist, "net": mean_net, "edge": beats})

    # save the most-recent-fold model as the staged candidate
    os.makedirs(OUT_DIR, exist_ok=True)
    model, fold = last_fold
    import joblib
    tag = f"{model_type}_{symbol}"
    torch.save(model.state_dict(), os.path.join(OUT_DIR, f"v2_{tag}.pth"))
    joblib.dump(fold["scaler"], os.path.join(OUT_DIR, f"scaler_{tag}.pkl"))
    meta = {"symbol": symbol, "model_type": model_type, "features": feats,
            "horizon": cfg["horizon"], "seq_length": cfg["seq_length"],
            "deadband_k": cfg["deadband_k"], "class_names": CLASS_NAMES,
            "use_extra_features": bool(cfg.get("use_extra_features")),
            "pooled_dir": pooled_dir, "pooled_p": pooled_p,
            "mean_err": mean_err, "mean_persist_err": mean_persist}
    with open(os.path.join(OUT_DIR, f"meta_{tag}.json"), "w") as f:
        json.dump(meta, f, indent=2)
    print(f"  saved -> models_v2/v2_{tag}.pth (+ scaler, meta)")
    return meta


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", choices=["lstm", "tft"], default="lstm")
    ap.add_argument("--symbol", default="BTCUSDT")
    ap.add_argument("--all", action="store_true", help="every coin x model")
    ap.add_argument("--horizon", type=int, default=None, help="override horizon (candles)")
    ap.add_argument("--smoke", action="store_true", help="tiny fast run to prove the code executes")
    args = ap.parse_args()

    cfg = dict(CONFIG)
    if args.horizon:
        cfg["horizon"] = args.horizon
    if args.smoke:
        cfg.update(seq_length=20, epochs=2, wf_folds=2, vol_window=96,
                   wf_test_frac=0.15, wf_val_frac=0.15, batch_size=64, smoke=True)

    combos = ([(m, s) for m in ("lstm", "tft") for s in SYMBOLS] if args.all
              else [(args.model, args.symbol)])
    for m, s in combos:
        run(m, s, cfg)


# ====== inference (pipeline/infer.py) ======
"""
Inference for the staged v2 (return + direction) models.

Produces a prediction in the SAME shape the live engine pushes to Supabase, so
swapping it into inference_orchestrator is mechanical once you've validated the
new models. Reconstruction is return-based (price = base * exp(pred_ret)), not
the legacy Bollinger-band mapping, and the trading signal comes from the
classifier (with a FLAT/abstain class) instead of a hand-tuned 0.1% threshold.

Load order mirrors training: same features, same seq_length, same scaler.
"""

import os
import json
import numpy as np
import torch


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def load_model(model_type: str, symbol: str):
    tag = f"{model_type}_{symbol}"
    meta = json.load(open(os.path.join(OUT_DIR, f"meta_{tag}.json")))
    import joblib
    scaler = joblib.load(os.path.join(OUT_DIR, f"scaler_{tag}.pkl"))
    n_feat = len(meta["features"])
    if model_type == "lstm":
        model = LSTMDual(n_feat)
    else:
        model = TFTDual(n_feat)
    model.load_state_dict(torch.load(os.path.join(OUT_DIR, f"v2_{tag}.pth"), map_location=DEVICE))
    model.to(DEVICE).eval()
    return model, scaler, meta


def predict(df_raw, model, scaler, meta):
    """df_raw: recent OHLCV+order-flow candles (>= seq_length+warmup rows).
    Returns a production-style dict for one model."""
    feats = meta["features"]
    seq = meta["seq_length"]
    df = compute_features(df_raw).dropna()
    if len(df) < seq:
        return None
    base_price = float(df["close"].iloc[-1])
    window = scaler.transform(df[feats].tail(seq).values)
    x = torch.tensor(window, dtype=torch.float32).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        pred_ret, logits = model(x)
    pred_ret = float(pred_ret.item())
    probs = torch.softmax(logits, -1).cpu().numpy().ravel()
    cls = int(probs.argmax())
    signal = {0: "SHORT", 1: "NEUTRAL", 2: "LONG"}[cls]
    pred_price = base_price * np.exp(pred_ret)
    return {
        "val": pred_ret,
        "price": float(pred_price),
        "change_pct": (np.exp(pred_ret) - 1) * 100,
        "signal": signal,
        "class_probs": {CLASS_NAMES[i]: float(probs[i]) for i in range(len(CLASS_NAMES))},
        "horizon_min": meta["horizon"] * 5,
    }

In [6]:
# ===== Cell 3: CONFIG — edit, then re-run THIS cell + Cell 4 =====
MODEL   = 'tft'          # 'lstm' or 'tft'
SYMBOL  = 'BTCUSDT'      # 'BTCUSDT' / 'ETHUSDT' / 'XRPUSDT'
HORIZON = 1              # candles ahead: 1=5m, 3=15m, 6=30m, 12=60m

# --- speed (smaller = faster; keep fast while just checking direction) ---
MAX_ROWS   = 150_000     # most-recent candles used. None = FULL history (decisive!)
WF_FOLDS   = 1           # walk-forward folds (1 = quickest; 3-5 = robust/slower)
EPOCHS     = 8
TRAIN_STEP = 2           # training-window stride (1 = every candle, 4 = 4x faster)

# --- model / signal levers ---
USE_EXTRA_FEATURES = True   # True = 26 features (volatility / order-flow / MTF)
SEQ_LENGTH = 60
DEADBAND_K = 0.33
CLS_W      = 1.0
LR         = 1e-3
DROPOUT    = 0.2

CFG = dict(CONFIG)
CFG.update(horizon=HORIZON, max_rows=MAX_ROWS, wf_folds=WF_FOLDS, epochs=EPOCHS,
           train_step=TRAIN_STEP, use_extra_features=USE_EXTRA_FEATURES,
           seq_length=SEQ_LENGTH, deadband_k=DEADBAND_K, cls_loss_weight=CLS_W,
           lr=LR, dropout=DROPOUT)
print('Experiment:', MODEL, SYMBOL, '| horizon', HORIZON*5, 'min |',
      ('26 features' if USE_EXTRA_FEATURES else '18 features'),
      '| max_rows', MAX_ROWS, '| folds', WF_FOLDS, '| epochs', EPOCHS)

Experiment: tft BTCUSDT | horizon 5 min | 26 features | max_rows 150000 | folds 1 | epochs 8


In [7]:
# ===== Cell 4: RUN ONE EXPERIMENT (auto-logged) — re-run after editing Cell 3 =====
result = run(MODEL, SYMBOL, CFG)


BTCUSDT  TFT  (horizon=5m, seq=60, features=26+extra, train_step=2, folds=1, max_rows=150000)
  data: 150000 rows  2025-01-08 -> 2026-06-13
  class balance: DOWN=48813, FLAT=52591, UP=48596
  fold 0: train 2025-01-08..2026-03-01 test 2026-04-22..2026-06-13 | n=14941 trades=6360 DIR= 51.3% (p=0.034) ERR=1.5099% vs persist 0.0797% net=-1.9bps
  --> POOLED OOS: DIR=51.3% (p=0.034, 3265/6360) ERR=1.5099% vs persist 0.0797%  net=-1.9bps  REAL EDGE: no
   logged -> /content/drive/MyDrive/CryptoProject/models_v2/experiments_log.csv  (appended; run #2)
  saved -> models_v2/v2_tft_BTCUSDT.pth (+ scaler, meta)


In [ ]:
# ===== Cell 5: FULL GRID SWEEP — every model x coin x horizon (auto-logged) =====
# Uses CFG from Cell 3, so it's a FAST scan when MAX_ROWS is small. For DECISIVE
# numbers, re-run the best config(s) with MAX_ROWS=None. Resilient: one failure
# won't abort the sweep, and every run is logged to Drive immediately.
# NOTE: saved .pth get overwritten per (model,symbol) across horizons -> results
# live in the log (Cell 6); deploy your chosen config via Cell 7.
import time, gc, torch

horizons = [1, 3, 6, 12]
symbols  = ['BTCUSDT', 'ETHUSDT', 'XRPUSDT']
models   = ['lstm', 'tft']

grid = [(h, s, m) for h in horizons for s in symbols for m in models]
done, t0 = 0, time.time()
for i, (h, s, m) in enumerate(grid, 1):
    el = time.time() - t0
    eta = (el / max(done, 1)) * (len(grid) - (i - 1)) if done else 0
    print(f"\n[{i}/{len(grid)}] {m.upper()} {s} {h*5}m   (elapsed {el/60:.0f}m, ~{eta/60:.0f}m left)")
    try:
        run(m, s, {**CFG, 'horizon': h})
        done += 1
    except Exception as e:
        print(f"   !! FAILED, skipping: {e}")
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
print(f"\nFinished {done}/{len(grid)} runs in {(time.time()-t0)/60:.0f} min. Run Cell 6 for the leaderboard.")

# --- Smaller targeted sweeps (uncomment instead if you don't want the full 24) ---
# for h in [1, 3, 6, 12]: run(MODEL, SYMBOL, {**CFG, 'horizon': h})        # horizons only
# for e in [False, True]: run(MODEL, SYMBOL, {**CFG, 'use_extra_features': e})  # features only

In [ ]:
# ===== Cell 6: LEADERBOARD — every logged experiment, best first =====
from IPython.display import display
log_csv = os.path.join(OUT_DIR, 'experiments_log.csv')
if not os.path.exists(log_csv):
    print('No experiments logged yet — run Cell 4 first.')
else:
    log = pd.read_csv(log_csv)
    log['_edge'] = (log['real_edge'] == 'YES').astype(int)
    log['_net'] = pd.to_numeric(log['net_bps'], errors='coerce').fillna(-9999)
    log = log.sort_values(['_edge', '_net', 'DIR'], ascending=False).drop(columns=['_edge', '_net'])
    print(f"{len(log)} runs logged. Full-history (decisive) runs have a blank max_rows.\n")
    full = log[log['max_rows'].isna()]
    if len(full):
        print('=== FULL-HISTORY RUNS (decisive) ==='); display(full)
    print('=== ALL RUNS ==='); display(log)

In [ ]:
# ===== Cell 7: TRAIN THE 6 DELIVERABLE MODELS — FULL HISTORY (run once; ~1.5-2 h) =====
# The decisive, report-grade run: BTC/ETH/XRP x TFT/LSTM at the product horizon,
# on the FULL history so the writeup can state "full history, 3 folds".
# train_step=2 (proven equal to 1) + wf_folds=3 keep it to ~1.5-2h instead of 4-6h.
# Crash-safe: each model logs + saves the moment it finishes, so if the runtime
# dies you just re-run for the remaining coins/models.
PRODUCT_HORIZON = 6        # candles: 6 = 30 min   (use 12 for 60 min)

FINAL = {**CFG, 'horizon': PRODUCT_HORIZON, 'max_rows': None, 'wf_folds': 3,
         'train_step': 2, 'epochs': 12, 'use_extra_features': True}

import time
t0 = time.time()
for s in ['BTCUSDT', 'ETHUSDT', 'XRPUSDT']:
    for m in ['tft', 'lstm']:
        run(m, s, FINAL)
print(f"\nAll 6 deliverable models trained in {(time.time()-t0)/60:.0f} min, saved to {OUT_DIR}.")
print(f"Horizon = {PRODUCT_HORIZON*5} min. Next: switch the engine + dashboard to this horizon.")

In [ ]:
# ===== Cell 8: INFERENCE TEST — load saved models and print a live-style signal =====
for s in ['BTCUSDT', 'ETHUSDT', 'XRPUSDT']:
    for m in ['lstm', 'tft']:
        try:
            mdl, sc, meta = load_model(m, s)
        except FileNotFoundError:
            continue
        df = pd.read_csv(os.path.join(DATA_DIR, f'{s}_5m_data.csv'))
        out = predict(df, mdl, sc, meta)
        print(f"{s} {m}: {out['signal']:7s} {out['change_pct']:+.3f}% "
              f"probs={ {k: round(v,2) for k,v in out['class_probs'].items()} } "
              f"h={out['horizon_min']}m feats={len(meta['features'])}")